In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import RobustScaler, MinMaxScaler

from catboost import CatBoostRegressor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Data collection

In [5]:
def load_data(ordinary_data, dir, data):
    path = f"{ordinary_data}/{dir}/{data}"
    with open(path, 'r', encoding='utf-8') as file:
        lines = file.readlines()[1:]  # Skip the first line (header)
    data_list = []
    for line in lines:
        values = list(map(float, line.split()))
        data_list.append(values)
    return data_list


def go_all_data(ordinary_data, dirs, nu_e_data):
    all_data = []
    for dir in dirs:
        data = load_data(ordinary_data, dir, nu_e_data)
        all_data.extend(data)
    return all_data



In [6]:
ordinary_data = "C:/Users/limon/OneDrive/Рабочий стол/Курсовая/abob_progs/ANN/DATA"

first_day = 12
last_day  =  17

dirs = [f"{day}-04-2026" for day in range(first_day, last_day + 1)]

nu_e_data = "fort.31"
forces_data = "fort.42"

In [7]:
X_data = go_all_data(ordinary_data, dirs, nu_e_data)
X_data = np.array(X_data)

Y_data = go_all_data(ordinary_data, dirs, forces_data)
Y_data = np.array(Y_data)


X_train, X_val_test, Y_train, Y_val_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)
X_val, X_test, Y_val, Y_test = train_test_split(X_val_test, Y_val_test, test_size=0.5, random_state=42)

In [8]:
def average_absolute_percentage_error(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)))

def average_squared_percentage_error(y_true, y_pred):
    #return (np.sum((y_true - y_pred) ** 2) / np.sum(y_true ** 2 )) ** 0.5
    return np.mean(np.sum((y_true - y_pred) ** 2, axis=1) / np.sum(y_true ** 2, axis=1)) ** 0.5

In [ ]:

X_scaler_R = RobustScaler()  # устойчив к выбросам
Y_scaler_R = MinMaxScaler(feature_range=(-1.5, 1.5))


#X_scaler = StandardScaler()
X_train_sc = X_scaler_R.fit_transform(X_train)
X_val_sc   = X_scaler_R.transform(X_val)
X_test_sc  = X_scaler_R.transform(X_test)

#Y_scaler = StandardScaler()

Y_train_sc = Y_scaler_R.fit_transform(Y_train)
Y_val_sc   = Y_scaler_R.transform(Y_val)
Y_test_sc  = Y_scaler_R.transform(Y_test)

# 2. Тензоры и DataLoader
train_dataset = TensorDataset(
    torch.tensor(X_train_sc, dtype=torch.float32),
    torch.tensor(Y_train_sc, dtype=torch.float32)
)
val_dataset = TensorDataset(
    torch.tensor(X_val_sc, dtype=torch.float32),
    torch.tensor(Y_val_sc, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
X_train_sc.shape, X_val_sc.shape, X_test_sc.shape

print(f"X_scaled range: [{X_train_sc.min():.3f}, {X_train_sc.max():.3f}]")
print(f"Y_scaled range: [{Y_train_sc.min():.3f}, {Y_train_sc.max():.3f}]")
print(f"X_scaled range: [{X_val_sc.min():.3f}, {X_val_sc.max():.3f}]")
print(f"Y_scaled range: [{Y_val_sc.min():.3f}, {Y_val_sc.max():.3f}]")
print(f"X_scaled range: [{X_test_sc.min():.3f}, {X_test_sc.max():.3f}]")
print(f"Y_scaled range: [{Y_test_sc.min():.3f}, {Y_test_sc.max():.3f}]")



# Try *Invertible* architecture

## Архитектура

In [13]:
class InverseLayers(nn.Module):

    def __init__(self, input_dim, output_dim, hidden_dims=[64, 128, 64], activation=nn.GELU, bn_momentum=0.1, norm_layers=False):
        super(InverseLayers, self).__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            if norm_layers:
                layers.append(nn.BatchNorm1d(h_dim, momentum=bn_momentum, eps=1e-5))
            layers.append(activation()) 
            prev_dim = h_dim
        Last_layer = nn.Linear(prev_dim, output_dim)
        # Инициализация в ноль: на старте s=0, t=0 => exp(s)=1 => v=u
        nn.init.zeros_(Last_layer.weight)
        nn.init.zeros_(Last_layer.bias)
        layers.append(Last_layer)        
        self.net = nn.Sequential(*layers)


    def forward(self, x):
        return self.net(x)

\begin{align}
    v_1 &= (u_2 + t_1(u_1)) \cdot exp(s_1(u_1)) & 
    u_1 &= (v_2 / exp(s_2(v_1))) - t_2(v_1) \\
    v_2 &= (u_1 + t_2(v_1)) \cdot exp(s_2(v_1)) & 
    u_2 &= (v_1 / exp(s_1(u_1))) - t_1(u_1)x
\end{align}

In [ ]:
# Обратимая нейросеть для преобразования между Enu и Forces
class InverseEnuToForcesANN(nn.Module):
    def __init__(self, 
                 inp1_dim=4, 
                 inp2_dim=4, 
                 layer_s1_dims=[32, 16], 
                 layer_s2_dims=[32, 16], 
                 layer_t1_dims=[32, 16], 
                 layer_t2_dims=[32, 16], 
                 s1_activation=nn.ReLU,
                 s2_activation=nn.ReLU,
                 t1_activation=nn.ReLU,
                 t2_activation=nn.ReLU,
                 clamp_range=(-5.0, 1.0),
                 bn_momentum=0.1,
                 norm_layers=False
                 ):
        super(InverseEnuToForcesANN, self).__init__()
        self.clamp_range = clamp_range
        self.inp1_dim = inp1_dim
        self.inp2_dim = inp2_dim
        self.clamp = clamp_range[1]

        self.s1 = InverseLayers(input_dim=inp2_dim, output_dim=inp1_dim, hidden_dims=layer_s1_dims, activation=s1_activation, bn_momentum=bn_momentum, norm_layers=norm_layers)
        self.t1 = InverseLayers(input_dim=inp2_dim, output_dim=inp1_dim, hidden_dims=layer_t1_dims, activation=t1_activation, bn_momentum=bn_momentum, norm_layers=norm_layers)

        self.s2 = InverseLayers(input_dim=inp1_dim, output_dim=inp2_dim, hidden_dims=layer_s2_dims, activation=s2_activation, bn_momentum=bn_momentum, norm_layers=norm_layers)
        self.t2 = InverseLayers(input_dim=inp1_dim, output_dim=inp2_dim, hidden_dims=layer_t2_dims, activation=t2_activation, bn_momentum=bn_momentum, norm_layers=norm_layers)


    def _clamp(self, x):
        return torch.clamp(x, min=self.clamp_range[0], max=self.clamp_range[1])

    def forward(self, x, inverse=False):
        if not inverse:
            u1 = x[:, :self.inp1_dim]
            u2 = x[:, self.inp1_dim:]

            #s1 = self._clamp(self.s1(u2))
            s1 = self.clamp * torch.tanh(self.s1(u2))
            t1 = self.t1(u2)
            v1 = u1 * torch.exp(s1) + t1
            
            #s2 = self._clamp(self.s2(v1))
            s2 = self.clamp * torch.tanh(self.s2(v1))
            t2 = self.t2(v1)
            v2 = u2 * torch.exp(s2) + t2
            
            return torch.cat([v1, v2], dim=1)
        else:
            v1 = x[:, :self.inp1_dim]
            v2 = x[:, self.inp1_dim:]
            
            s2 = self.clamp * torch.tanh(self.s2(v1))
            #s2 = self._clamp(self.s2(v1))
            t2 = self.t2(v1)
            u2 = (v2 - t2) * torch.exp(-s2)
            
            s1 = self.clamp * torch.tanh(self.s1(u2))
            #s1 = self._clamp(self.s1(u2))
            t1 = self.t1(u2)
            u1 = (v1 - t1) * torch.exp(-s1)
            
            return torch.cat([u1, u2], dim=1)

In [ ]:
class MultiInvertibleANN(nn.Module):
    def __init__(
        self, 
        inp1_dim=4, 
        inp2_dim=4, 
        layer_s1_dims=[32, 16], 
        layer_s2_dims=[32, 16], 
        layer_t1_dims=[32, 16], 
        layer_t2_dims=[32, 16], 
        s1_activation=nn.ReLU,
        s2_activation=nn.ReLU,
        t1_activation=nn.ReLU,
        t2_activation=nn.ReLU,
        inv_layers=3,
        clamp_range=(-1.0, 1.0),
        bn_momentum=0.1,
        norm_layers=False,
    ):
        
        # Фиксированная случайная перестановка для каждого слоя
        self.permutations = []
        self.inv_permutations = []
        for _ in range(inv_layers):
            perm = torch.randperm(inp1_dim + inp2_dim)
            self.permutations.append(perm)
            self.inv_permutations.append(torch.argsort(perm))
        
        # Слои
        super(MultiInvertibleANN, self).__init__()
        self.inp1_dim = inp1_dim
        self.inp2_dim = inp2_dim
        self.layers = nn.ModuleList()
        for _ in range(inv_layers):
            self.layers.append(InverseEnuToForcesANN(
                inp1_dim=inp1_dim, 
                inp2_dim=inp2_dim, 
                layer_s1_dims=layer_s1_dims, 
                layer_s2_dims=layer_s2_dims, 
                layer_t1_dims=layer_t1_dims, 
                layer_t2_dims=layer_t2_dims, 
                s1_activation=s1_activation,
                s2_activation=s2_activation,
                t1_activation=t1_activation,
                t2_activation=t2_activation,
                clamp_range=clamp_range,
                bn_momentum=bn_momentum,
                norm_layers=norm_layers
            ))
            

    def _permute(self, x, layer_idx):
        """Прямая перестановка для слоя layer_idx"""
        perm = self.permutations[layer_idx].to(x.device)
        return x[:, perm]
    
    def _permute_inv(self, x, layer_idx):
        """Обратная перестановка для слоя layer_idx"""
        inv_perm = self.inv_permutations[layer_idx].to(x.device)
        return x[:, inv_perm]

    def forward(self, x, inverse=False):
        if not inverse:
            for idx, layer in enumerate(self.layers):
                x = layer(x, inverse=False)
                x = self._permute(x, idx)  # ← применяем перестановку слоя
        else:
            for idx in reversed(range(len(self.layers))):
                x = self._permute_inv(x, idx)  # ← обратная перестановка
                x = self.layers[idx](x, inverse=True)
        return x

In [16]:
model_inv = InverseEnuToForcesANN()
model_inv

InverseEnuToForcesANN(
  (s1): InverseLayers(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=16, bias=True)
      (3): ReLU()
      (4): Linear(in_features=16, out_features=4, bias=True)
    )
  )
  (t1): InverseLayers(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=16, bias=True)
      (3): ReLU()
      (4): Linear(in_features=16, out_features=4, bias=True)
    )
  )
  (s2): InverseLayers(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=16, bias=True)
      (3): ReLU()
      (4): Linear(in_features=16, out_features=4, bias=True)
    )
  )
  (t2): InverseLayers(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_fea

In [17]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_multi = MultiInvertibleANN(inv_layers=3).to(device)
model_multi

MultiInvertibleANN(
  (layers): ModuleList(
    (0-2): 3 x InverseEnuToForcesANN(
      (s1): InverseLayers(
        (net): Sequential(
          (0): Linear(in_features=4, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=16, bias=True)
          (3): ReLU()
          (4): Linear(in_features=16, out_features=4, bias=True)
        )
      )
      (t1): InverseLayers(
        (net): Sequential(
          (0): Linear(in_features=4, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=16, bias=True)
          (3): ReLU()
          (4): Linear(in_features=16, out_features=4, bias=True)
        )
      )
      (s2): InverseLayers(
        (net): Sequential(
          (0): Linear(in_features=4, out_features=32, bias=True)
          (1): ReLU()
          (2): Linear(in_features=32, out_features=16, bias=True)
          (3): ReLU()
          (4): Linear(in_features=16, out_features=4, bias=True)


In [18]:
pred = model_multi(torch.tensor(X_train_sc[0:1], dtype=torch.float32).to(device), inverse=False)
pred,  model_multi(pred, inverse=True), X_train_sc[0:1]

(tensor([[-0.6763, -0.5600, -0.9567,  0.2498,  0.0029, -0.7730,  0.1845, -0.4409]],
        device='cuda:0', grad_fn=<IndexBackward0>),
 tensor([[-0.4409,  0.1845,  0.2498,  0.0029, -0.9567, -0.6763, -0.5600, -0.7730]],
        device='cuda:0', grad_fn=<CatBackward0>),
 array([[-0.44087379,  0.1845407 ,  0.24977408,  0.0029381 , -0.95671994,
         -0.67625192, -0.56004979, -0.77296613]]))

## Функции обучения

In [91]:
def robust_inverse_loss(model, x, y, sigma_noise=0.01):
    # Прямой проход
    y_pred = model(x, inverse=False)
    loss_fwd = F.mse_loss(y_pred, y)
    
    # Обратный проход с шумом 
    y_noisy = y + torch.randn_like(y) * sigma_noise
    x_rec   = model(y_noisy, inverse=True)
    loss_inv = F.mse_loss(x_rec, x)
    
    return loss_fwd, loss_inv

def train_bijection(
    model, 
    train_loader, 
    val_loader, 
    num_epochs=150, 
    learning_rate=1e-3, 
    inv_coef=0.01,
    patience=25,
    inv_epoch_limit=50,
    inv_loss_limit=1e-3,
    device='cpu', 
    history = {'train_loss': [], 'val_loss': [], 'train_fwd': [], 'val_fwd': [], 'train_inv': [], 'val_inv': []}
):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=patience)

    
    for epoch in range(num_epochs):
        model.train()
        t_fwd, t_inv = 0.0, 0.0
        
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            
            loss_fwd, loss_inv = robust_inverse_loss(model, x_batch, y_batch, sigma_noise=0.01)
            loss = loss_fwd
            if epoch >= inv_epoch_limit or t_fwd < inv_loss_limit:
                loss += inv_coef * loss_inv
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
            optimizer.step()
            
            t_fwd += loss_fwd.item() * x_batch.size(0)
            t_inv += loss_inv.item() * x_batch.size(0)
            
        model.eval()
        v_fwd, v_inv = 0.0, 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                # На валидации шум и регуляризация не нужны
                f, i = robust_inverse_loss(model, x_batch, y_batch, sigma_noise=0.01)
                v_fwd += f.item() * x_batch.size(0)
                v_inv += i.item() * x_batch.size(0)
                
        n_tr, n_val = len(train_loader.dataset), len(val_loader.dataset)
        history['train_loss'].append((t_fwd + t_inv) / n_tr)
        history['val_loss'].append((v_fwd + v_inv) / n_val)
        history['train_fwd'].append(t_fwd / n_tr)
        history['val_fwd'].append(v_fwd / n_val)
        history['train_inv'].append(t_inv / n_tr)
        history['val_inv'].append(v_inv / n_val)
        
        scheduler.step(history['val_loss'][-1])
        if epoch % 2 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch:3d} | Val Loss: {history['val_loss'][-1]:.4e} | "
                  f"Fwd: {history['val_fwd'][-1]:.4e} | Inv: {history['val_inv'][-1]:.4e} | "
                  f"Epoch {epoch:3d} | Train Loss: {history['train_loss'][-1]:.4e} | "
                  f"Fwd: {history['train_fwd'][-1]:.4e} | Inv: {history['train_inv'][-1]:.4e} | LR: {lr:.2e}")
                  
    return model, history

## Тренировка


In [125]:
S_layers = [32]
T_layers = [128, 64, 32]
#T_layers = [32, 256, 128,64, 32]
activation_t = nn.ReLU
activation_s = nn.ELU

model_multi = MultiInvertibleANN(
        inp1_dim=4,     
        inp2_dim=4, 
        layer_s1_dims=S_layers,
        layer_s2_dims=S_layers,
        layer_t1_dims=T_layers,
        layer_t2_dims=T_layers,
        s1_activation=activation_s,
        s2_activation=activation_s,
        t1_activation=activation_t,
        t2_activation=activation_t,
        inv_layers=6,
        clamp_range=(-1, 0.5),
        bn_momentum=0.1,
        #norm_layers=True
        ).to(device)

history = {'train_loss': [], 'val_loss': [], 'train_fwd': [], 'val_fwd': [], 'train_inv': [], 'val_inv': []}

In [ ]:
model_multi, history = train_bijection( model_multi, 
                                        train_loader, 
                                        val_loader, 
                                        num_epochs=150, 
                                        learning_rate=1e-3, 
                                        inv_coef=0.9,
                                        patience=5,
                                        inv_epoch_limit=1,
                                        inv_loss_limit=1e-3,
                                        device=device, 
                                        history=history) 

## Results 

In [ ]:
model_multi.eval()
pred_y = model_multi(torch.tensor(X_test_sc, dtype=torch.float32).to(device), inverse=False)
pred_y[:2], Y_test_sc[:2]

(tensor([[ 0.5255, -0.4033,  0.0109,  0.0701,  0.1737,  0.0494, -0.5156,  0.2889],
         [ 0.5488, -0.2210, -0.3920,  0.0502,  0.0346, -0.3221, -0.5500,  0.7815]],
        device='cuda:0', grad_fn=<SliceBackward0>),
 array([[ 0.5529631 , -0.47800673, -0.00763294,  0.1583788 ,  0.16818083,
          0.08350032, -0.43059796,  0.24028845],
        [ 0.5529631 , -0.15955573, -0.41720293,  0.07898749,  0.08195982,
         -0.3194496 , -0.65410814,  0.740569  ]]))

In [ ]:
real_pred_y = Y_scaler_R.inverse_transform(pred_y.cpu().detach().numpy())
real_y = Y_scaler_R.inverse_transform(Y_test_sc)

mse = mean_squared_error(real_pred_y, real_y)
mae = mean_absolute_error(real_pred_y, real_y)
r2 = r2_score(real_pred_y, real_y)

print(f"Mean Squared Error: {mse:.2e}")
print(f"Mean Absolute Error: {mae:.2e}")
print(f"R^2 Score: {r2:.4f}")
real_pred_y[:2], real_y[:2]

Mean Squared Error: 2.53e-15
Mean Absolute Error: 3.78e-08
R^2 Score: 0.9968


(array([[-1.75537878e-08, -5.10246946e-06, -8.70119038e-06,
         -1.03655429e-05, -1.03761686e-05, -9.08716174e-06,
         -6.00959811e-06, -4.03446990e-07],
        [-2.67590394e-09, -4.88999058e-06, -8.89186140e-06,
         -1.03861094e-05, -1.05247955e-05, -9.31016075e-06,
         -6.04165098e-06,  3.65315600e-08]], dtype=float32),
 array([[ 0.00000000e+00, -5.18949704e-06, -8.70993904e-06,
         -1.02745108e-05, -1.03821126e-05, -9.06668059e-06,
         -5.93040405e-06, -4.46855903e-07],
        [ 0.00000000e+00, -4.81845184e-06, -8.90378600e-06,
         -1.03563851e-05, -1.04742431e-05, -9.30854715e-06,
         -6.13858678e-06,  0.00000000e+00]]))

In [ ]:
average_absolute_percentage_error(real_y, real_pred_y), average_squared_percentage_error(real_y, real_pred_y)

(np.float64(0.0060533315317640776), np.float64(0.006834703919955625))

In [ ]:
real_pred = X_scaler_R.inverse_transform(pred_x.cpu().detach().numpy())
real_x = X_scaler_R.inverse_transform(X_test_sc)

mse = mean_squared_error(real_pred, real_x)
mae = mean_absolute_error(real_pred, real_x)
r2 = r2_score(real_pred, real_x)

print(f"Mean Squared Error: {mse:.2e}")
print(f"Mean Absolute Error: {mae:.2e}")
print(f"R^2 Score: {r2:.4f}")
real_pred[:2], real_x[:2]

Mean Squared Error: 6.96e-02
Mean Absolute Error: 1.55e-01
R^2 Score: -22.3001


(array([[1.1957848 , 0.33348686, 1.0787648 , 0.3482533 , 1.8239899 ,
         0.3288281 , 1.0452788 , 0.3299305 ],
        [1.6083484 , 0.3358258 , 1.6360915 , 0.33654585, 0.771906  ,
         0.33006257, 2.2211423 , 0.32622972]], dtype=float32),
 array([[1.137562 , 0.4091283, 0.8042313, 0.30784  , 1.456046 , 0.3484014,
         0.9803315, 0.4404913],
        [1.82822  , 0.4322203, 1.034358 , 0.3024453, 0.8126647, 0.403428 ,
         2.259254 , 0.3110213]]))

In [ ]:
average_absolute_percentage_error(real_x, real_pred), average_squared_percentage_error(real_x, real_pred)

(np.float64(0.1717359967633015), np.float64(0.22735302951090408))

In [ ]:
def show_all_history(history, pairs=[("train_loss", "val_loss")], save=False, 
                     file_names=["history.png"], titles=["Training History"], 
                     path="C:/Users/limon/OneDrive/Рабочий стол/Курсовая/my_code/graphics"):
    for (train_key, val_key), title, fname in zip(pairs, titles, file_names):
        plt.figure(figsize=(12, 5))
        plt.plot(history[train_key], label=train_key)
        plt.plot(history[val_key], label=val_key)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(title)
        plt.legend()
        plt.grid()
        
        if save:
            file_path = Path(path) / fname
            file_path.parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(file_path, dpi=300, bbox_inches='tight')
            print(f"Plot saved to: {file_path}")
        plt.show()